# MathNet CNN — Math Problem Detection & Understanding
Uses the `ShadenA/MathNet` dataset with a CNN-based classifier to categorize math problems by topic.

## 1. Install Dependencies

In [ ]:
!pip install datasets transformers torch torchvision matplotlib scikit-learn seaborn pandas numpy

## 2. Load the Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("ShadenA/MathNet", "all")
print(ds)
print("\nExample entry:")
print(ds["train"][0])

## 3. Explore the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_train = pd.DataFrame(ds["train"])
print("Columns:", df_train.columns.tolist())
print("Shape:", df_train.shape)
df_train.head()

In [ ]:
# Identify the label / topic column — adjust 'label' if the column name differs
label_col = [c for c in df_train.columns if c in ("label", "topic", "category", "subject", "type")]
label_col = label_col[0] if label_col else df_train.columns[-1]
print("Using label column:", label_col)

plt.figure(figsize=(12, 5))
df_train[label_col].value_counts().plot(kind="bar", color="steelblue")
plt.title("Distribution of Math Problem Categories")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Preprocess — Tokenise Text for 1-D CNN

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ------------------------------------------------------------------
# Identify the text column (problem statement)
# ------------------------------------------------------------------
text_col = [c for c in df_train.columns if c in ("problem", "question", "text", "Problem", "Question")]
text_col = text_col[0] if text_col else df_train.columns[0]
print("Text column:", text_col, "| Label column:", label_col)

# Encode labels to integers
le = LabelEncoder()
y_all = le.fit_transform(df_train[label_col].astype(str))
num_classes = len(le.classes_)
print(f"Classes ({num_classes}):", le.classes_)

# Tokenise the problem text
MAX_WORDS = 20_000
MAX_LEN   = 128

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df_train[text_col].astype(str))

X_all = pad_sequences(
    tokenizer.texts_to_sequences(df_train[text_col].astype(str)),
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)
print("X shape:", X_all.shape, "| y shape:", y_all.shape)

## 5. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print("Train:", X_train.shape, "| Val:", X_val.shape)

## 6. Build the 1-D CNN Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

EMBED_DIM = 64

def build_cnn(vocab_size, embed_dim, seq_len, num_classes):
    inp = layers.Input(shape=(seq_len,))
    x   = layers.Embedding(vocab_size, embed_dim)(inp)

    # Parallel convolution branches — capture n-gram patterns of sizes 3, 4, 5
    branches = []
    for k in (3, 4, 5):
        b = layers.Conv1D(128, k, activation="relu", padding="same")(x)
        b = layers.GlobalMaxPooling1D()(b)
        branches.append(b)

    merged = layers.Concatenate()(branches)        # 384-dim
    merged = layers.Dropout(0.4)(merged)
    merged = layers.Dense(128, activation="relu")(merged)
    merged = layers.Dropout(0.3)(merged)
    out    = layers.Dense(num_classes, activation="softmax")(merged)

    model = models.Model(inp, out)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_cnn(MAX_WORDS, EMBED_DIM, MAX_LEN, num_classes)
model.summary()

## 7. Train

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=2, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=callbacks
)

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, title in zip(
    axes,
    [("accuracy", "val_accuracy"), ("loss", "val_loss")],
    ["Accuracy", "Loss"]
):
    ax.plot(history.history[metric[0]], label="Train")
    ax.plot(history.history[metric[1]], label="Val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()

plt.tight_layout()
plt.show()

## 9. Evaluation — Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(model.predict(X_val), axis=1)

print(classification_report(y_val, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 10. Predict on a New Problem

In [ ]:
def predict_topic(problem_text: str) -> dict:
    """Return predicted topic and confidence scores for a math problem."""
    seq = pad_sequences(
        tokenizer.texts_to_sequences([problem_text]),
        maxlen=MAX_LEN, padding="post", truncating="post"
    )
    probs     = model.predict(seq, verbose=0)[0]
    top_idx   = np.argsort(probs)[::-1][:3]
    return {
        "predicted_topic": le.classes_[top_idx[0]],
        "confidence":      float(probs[top_idx[0]]),
        "top_3": [
            {"topic": le.classes_[i], "score": float(probs[i])}
            for i in top_idx
        ]
    }

# --- try it ---
sample = "Find the derivative of f(x) = 3x^2 + 2x - 5"
result = predict_topic(sample)
print(f"Problem: {sample}")
print(f"Predicted topic : {result['predicted_topic']}  ({result['confidence']:.2%} confidence)")
print("Top 3 predictions:")
for t in result["top_3"]:
    print(f"  {t['topic']:<30} {t['score']:.2%}")

## 11. Save the Model

In [ ]:
import pickle, os

os.makedirs("saved_model", exist_ok=True)
model.save("saved_model/mathnet_cnn.keras")

with open("saved_model/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
with open("saved_model/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model, tokenizer, and label encoder saved to saved_model/")